In [2]:
import pandas as pd
import json

# 1. 파일 로드
with open('coupang_raw.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

with open('coupang_processed.json', 'r', encoding='utf-8') as f:
    processed_data = json.load(f)

# 2. Pandas DataFrame으로 변환 및 정규화(Flatten)
df_raw = pd.DataFrame(raw_data)
# metadata 필드가 중첩되어 있으므로 json_normalize를 사용해 컬럼으로 펼칩니다.
df_processed = pd.json_normalize(processed_data)

# 3. 비교를 위한 데이터 병합 (id 기준)
# 원본의 answer와 전처리된 document를 대조하기 위해 컬럼명을 변경합니다.
df_raw = df_raw.rename(columns={'answer': 'raw_answer', 'question': 'raw_question'})
comparison_df = pd.merge(
    df_raw[['id', 'raw_question', 'raw_answer']], 
    df_processed, 
    on='id'
)

# 4. 보기 편하게 컬럼 순서 재배치
cols = [
    'id', 'raw_question', 'document', 'metadata.category', 
    'metadata.subject', 'metadata.tags', 'raw_answer'
]
comparison_df = comparison_df[cols]

# 5. 주피터 노트북에서 가독성을 높이기 위한 스타일 설정
# 텍스트가 잘리지 않도록 길이를 조절합니다.
pd.set_option('display.max_colwidth', None)

# 상위 5개 항목 출력
comparison_df.head()

,id,raw_question,document,metadata.category,metadata.subject,metadata.tags,raw_answer
0,1,제 24 조 1항 (청약철회 등),[청약철회 원칙] 제24조 제1항: 회원은 계약 서면 수령(또는 공급 시작)일로부터 7일 이내에 자유롭게 청약을 철회(환불)할 수 있다.,원칙,회원,"[청약철회, 환불권리, 7일이내]","회사와 상품 등의 구매에 관한 계약을 체결한 회원은 『전자상거래 등에서의 소비자보호에 관한 법률』 제13조 제2항에 따른 계약내용에 관한 서면을 받은 날(그 서면을 받은 때보다 상품 등의 공급이 늦게 이루어진 경우에는 상품 등을 공급받거나 상품 등의 공급이 시작된 날을 말함)부터 7일 이내에는 청약을 철회할 수 있습니다. 다만, 청약철회에 관하여 『전자상거래 등에서의 소비자보호에 관한 법률』에 달리 정함이 있는 경우에는 동 법 규정에 따릅니다."
1,2,제 24 조 2항 (청약철회 등),[청약철회 제한] 제24조 제2항: 상품 배송 후 특정 사유(각 호 참조)에 해당하면 회원은 반품 및 교환을 요청할 수 없다.,예외-안내,회원,"[반품불가, 교환불가, 제한사항]",회원은 상품 등을 배송 받은 경우 다음 각 호에 해당하는 경우에는 반품 및 교환을 할 수 없습니다.
2,3,제 24 조 2항 1조 (청약철회 등),"[청약철회 예외] 제24조 제2항 제1호: 회원 책임으로 상품이 멸실/훼손된 경우 반품 불가. (단, 내용 확인을 위한 포장 훼손은 제외)",예외-상세,회원,"[상품훼손, 멸실, 포장훼손제외]","회원에게 책임 있는 사유로 상품 등이 멸실 또는 훼손된 경우(다만, 상품 등의 내용을 확인하기 위하여 포장 등을 훼손한 경우는 제외함)"
3,4,제 24 조 2항 2조 (청약철회 등),[청약철회 예외] 제24조 제2항 제2호: 회원의 사용 또는 일부 소비로 상품 가치가 현저히 감소한 경우 반품 불가.,예외-상세,회원,"[가치감소, 사용흔적, 소비]",회원의 사용 또는 일부 소비에 의하여 상품 등의 가치가 현저히 감소한 경우
4,5,제 24 조 2항 3조 (청약철회 등),[청약철회 예외] 제24조 제2항 제3호: 시간 경과로 재판매가 곤란할 정도로 상품 가치가 현저히 감소한 경우 반품 불가.,예외-상세,회원,"[시간경과, 재판매곤란, 가치하락]",시간의 경과에 의하여 재판매가 곤란할 정도로 상품 등의 가치가 현저히 감소한 경우


In [3]:
import json
import difflib
from IPython.display import HTML, display

# 1. 데이터 로드
with open('coupang_raw.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)
with open('coupang_processed.json', 'r', encoding='utf-8') as f:
    processed_data = json.load(f)

# 비교하고 싶은 샘플 개수 설정 (전체는 너무 길 수 있으니 상위 5개만)
num_samples = 5
diff_html = ""

html_diff = difflib.HtmlDiff()

for i in range(num_samples):
    raw_text = raw_data[i]['answer']
    proc_text = processed_data[i]['document']
    
    # 조항 번호 정보를 제목으로 추가
    header = f"<h3>ID: {raw_data[i]['id']} | {raw_data[i]['question']}</h3>"
    
    # 두 텍스트를 리스트 형태로 넣어야 비교가 가능합니다.
    # 문장 단위로 쪼개서 넣으면 더 보기 좋습니다.
    diff = html_diff.make_table(
        raw_text.split('. '), 
        proc_text.split('. '), 
        fromdesc='원본 (Raw)', 
        todesc='가공 (Processed)'
    )
    diff_html += header + diff + "<br><hr>"

# 2. 주피터 노트북에 결과 렌더링
display(HTML(diff_html))